# Predicted Epitopes of Trypanosoma cruzi  based on Phage Display Data Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.etbd-dths/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()

print(f"Dataset: {dataset.metadata.name}\n\nDescription: {dataset.metadata.description}\n")
print(f"Published on: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")
print(f"Citation: {dataset.metadata.citeAs}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each entity (record set, field, column, etc.) is referenced by its `@id` as per Croissant specification.

In [ ]:
# List all record sets and their fields
def print_record_sets(ds):
    print("Available Record Sets (by @id):\n-------------------------------")
    for rs in ds.metadata.record_sets:
        print(f"@id: {rs.id}, name: {rs.name}, description: {getattr(rs, 'description', '')}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
        print()

print_record_sets(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Get list of all record set @id values
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]

# Prepare a dictionary for DataFrames
dataframes = {}

# Load each record set into a separate DataFrame
for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set {record_set_id}")
        if len(df.columns) <= 10:
            print(f"Columns: {df.columns.tolist()}")
        else:
            print(f"First 10 columns: {df.columns[:10].tolist()}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {str(e)}")
        continue

# Example: Print first few rows of the first available record set
if record_sets_ids:
    rs0 = record_sets_ids[0]
    print(f"\nFirst few rows of record set {rs0}:")
    display(dataframes[rs0].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

> **Note:** For demonstration, we use the first record set and search for a numeric field.

In [ ]:
# Pick a record set and a numeric field for analysis
import numpy as np

# Automatically find the first numeric field
selected_record_set_id = None
numeric_field_id = None

for rs in dataset.metadata.record_sets:
    for field in rs.fields:
        if field.data_type in ["Float", "Integer", "Number"]:
            selected_record_set_id = rs.id
            numeric_field_id = field.id
            print(f"Using record set {selected_record_set_id}, numeric field {numeric_field_id}")
            break
    if selected_record_set_id:
        break

if not selected_record_set_id or selected_record_set_id not in dataframes or numeric_field_id not in dataframes[selected_record_set_id].columns:
    print("No suitable numeric field found for EDA.")
else:
    df = dataframes[selected_record_set_id]
    # Handle missing or non-numeric values
    df = df.copy()
    df = df[df[numeric_field_id].apply(lambda x: pd.notnull(x) and np.issubdtype(type(x), np.number))]

    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field
    group_field = None
    for field in dataset.metadata.record_sets[0].fields:
        if field.data_type == "Text" and field.id in df.columns:
            group_field = field.id
            break

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"Grouped data by {group_field} (mean of {numeric_field_id}):")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_field_id and selected_record_set_id in dataframes:
    df = dataframes[selected_record_set_id]
    if numeric_field_id in df.columns and not df[numeric_field_id].empty:
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_field_id].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field_id} in Record Set {selected_record_set_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()
    else:
        print(f"No data available in field {numeric_field_id}")
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to use the `mlcroissant` library to load, explore, and analyze a Croissant-structured FAIR dataset using unique `@id` references for all entities. We programmatically explored all record sets and fields, performed basic data cleaning and normalization, and visualized numeric distributions. For further analysis, consider exploring additional record sets, relationships across entities by their `@id`, and leveraging advanced analytics techniques.